# Kaggle to Bronze data

In [4]:
import os
import duckdb
import kagglehub

base_project_path = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion"
bronze_root_path = os.path.join(base_project_path, "data", "bronze")

datasets_to_ingest = [
    {
        "kaggle_handle": "avikumart/analytics-vidhya-nov22-insurance-claims-dataset",
        "target_subfolder": "analytics_vidhya_claims"
    },
    {
        "kaggle_handle": "apoorvasharma03/vehicle-insurance-dataset",
        "target_subfolder": "vehicle_insurance"
    }
]

con = duckdb.connect()

for ds in datasets_to_ingest:
    print(f"\n--- Processing Dataset: {ds['kaggle_handle']} ---")
    dataset_path = kagglehub.dataset_download(ds["kaggle_handle"])
    dataset_bronze_dir = os.path.join(bronze_root_path, ds["target_subfolder"])
    os.makedirs(dataset_bronze_dir, exist_ok=True)
    
    for filename in os.listdir(dataset_path):
        if filename.endswith(".csv"):
            file_full_path = os.path.join(dataset_path, filename).replace("\\", "/")
            table_name = filename.split("_")[0].lower()
            output_file = os.path.join(dataset_bronze_dir, f"{table_name}.parquet").replace("\\", "/")
            
            print(f"Ingesting {filename} -> {output_file}")
            
            con.execute(f"""
                COPY (
                    SELECT *, 
                           CURRENT_TIMESTAMP AS _ingestion_timestamp,
                           '{file_full_path}' AS _source_file
                    FROM read_csv_auto('{file_full_path}')
                ) TO '{output_file}' (FORMAT PARQUET);
            """)

print("\nBronze ingestion with DuckDB complete!")


--- Processing Dataset: avikumart/analytics-vidhya-nov22-insurance-claims-dataset ---
Ingesting sample_submission_KvRh9Sx.csv -> C:/Users/PC 12/Motor-Insurance-Quote-to-Policy-Conversion/data/bronze/analytics_vidhya_claims/sample.parquet
Ingesting test_zo1G9sv.csv -> C:/Users/PC 12/Motor-Insurance-Quote-to-Policy-Conversion/data/bronze/analytics_vidhya_claims/test.parquet
Ingesting train_qWM28Yl.csv -> C:/Users/PC 12/Motor-Insurance-Quote-to-Policy-Conversion/data/bronze/analytics_vidhya_claims/train.parquet

--- Processing Dataset: apoorvasharma03/vehicle-insurance-dataset ---
Ingesting Vehicle_Insurance.csv -> C:/Users/PC 12/Motor-Insurance-Quote-to-Policy-Conversion/data/bronze/vehicle_insurance/vehicle.parquet

Bronze ingestion with DuckDB complete!
